In [57]:
pip install shap

In [58]:
import shap
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

In [59]:
df = sns.load_dataset("titanic")

df = df[['survived','pclass','sex','age','fare','sibsp','parch','embarked']]
df.head()

,survived,pclass,sex,age,fare,sibsp,parch,embarked
0,0,3,male,22.0,7.2500,1,0,S
1,1,1,female,38.0,71.2833,1,0,C
2,1,3,female,26.0,7.9250,0,0,S
3,1,1,female,35.0,53.1000,1,0,S
4,0,3,male,35.0,8.0500,0,0,S


In [60]:
# Handle missing values
df['age'].fillna(df['age'].median(), inplace=True)
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)

# Encode categorical variables
le = LabelEncoder()
df['sex'] = le.fit_transform(df['sex'])
df['embarked'] = le.fit_transform(df['embarked'])

/tmp/ipykernel_320/1716769451.py:2: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.



/tmp/ipykernel_320/1716769451.py:3: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or d

In [61]:
X = df.drop('survived', axis=1)
y = df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [63]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.7988826815642458


In [64]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# For binary classification, class 1 (Survived)
shap_vals = shap_values[1]

In [65]:
feature_importance = np.abs(shap_values[:, :, 1]).mean(axis=0)

importance_df = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance": feature_importance
}).sort_values(by="Importance", ascending=False)

importance_df

,Feature,Importance
1,sex,0.242443
0,pclass,0.136670
3,fare,0.032850
2,age,0.028978
6,embarked,0.014331
4,sibsp,0.007725
5,parch,0.001041


In [66]:
fig = px.bar(
    importance_df,
    x="Importance",
    y="Feature",
    orientation="h",
    title="Global Feature Importance (SHAP)",
)

fig.update_layout(yaxis=dict(categoryorder='total ascending'))
fig.show()

In [67]:
shap_df = pd.DataFrame(shap_values[:, :, 1], columns=X_test.columns)

melted_df = shap_df.melt(var_name="Feature", value_name="SHAP Value")

fig = px.box(
    melted_df,
    x="Feature",
    y="SHAP Value",
    title="SHAP Value Distribution per Feature"
)

fig.update_layout(xaxis_tickangle=45)
fig.show()

In [68]:
i = 5

single_shap = shap_vals[i]
base_value = explainer.expected_value[1]
features = X_test.iloc[i]

In [69]:
i = 5 # Assuming 'i' is the intended index of the sample for local explanation
single_shap = shap_values[i, :, 1] # Correctly extracts SHAP values for instance 'i', all features, class 1

local_df = pd.DataFrame({
    "Feature": X_test.columns,
    "SHAP Contribution": single_shap
}).sort_values(by="SHAP Contribution", ascending=False)

In [70]:
local_df = pd.DataFrame({
    "Feature": X_test.columns,
    "SHAP Contribution": single_shap
}).sort_values(by="SHAP Contribution", ascending=False)

In [71]:
fig = px.bar(
    local_df,
    x="SHAP Contribution",
    y="Feature",
    orientation="h",
    title="Local SHAP Explanation (Passenger Index 5)",
    color="SHAP Contribution"
)

fig.show()

In [72]:
dependence_df = pd.DataFrame({
    "Age": X_test["age"],
    "SHAP Value": shap_df["age"]
})

fig = px.scatter(
    dependence_df,
    x="Age",
    y="SHAP Value",
    title="SHAP Dependence Plot for Age",
    trendline="ols"
)

fig.show()